<h1>Chapter 7 - Evaluation</h1>
<i>Measuring whether your `TinyAgent` actually works.</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 7 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>

### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter.

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to **Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM - `Gemma 4`

We use the same LLM that we used previously, namely the Gemma 4 E4B model with native tool calling and reasoning capabilities. Note that it will also be used to judge itself, as we will explore in the LLM-as-a-judge section.

In [1]:
from openai import OpenAI
from illustrated_agents.chapters.ch2 import LLM

client = OpenAI(base_url="http://localhost:11434/v1/", api_key="no_key")
llm = LLM(model="gemma4:e4b", client=client, think=True)

## 2 - Building the `Evaluator`

In [2]:
from dataclasses import dataclass
from typing import Callable

# Type hint for the scorers
##  (prediction: str, example: dict) -> bool | float
Scorer = Callable[[str, dict], bool | float]

@dataclass
class Benchmark:
    name: str
    examples: list[dict]
    scorer: Callable

In [3]:
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import NativeTools
from illustrated_agents.chapters.ch6 import NativeReAct, TinyAgent


class Evaluator:
    """Run a TinyAgent over a Benchmark and aggregate the results."""

    def __init__(self, create_agent: Callable):
        """Initialize with a function that creates a new agent instance."""
        self.create_agent = create_agent

    def run(self, benchmark: Benchmark) -> dict:
        """Run the agent on each example in the benchmark and score the results."""

        # Run each example and collect results
        results = []
        for example in benchmark.examples:
            agent = self.create_agent()
            prediction = agent.run(example["task"]) or ""
            passed = benchmark.scorer(prediction, example)
            results.append(
                {
                    "prediction": prediction,
                    "passed": passed,
                }
            )

        # Aggregate pass rate
        if results:
            pass_rate = sum(result["passed"] for result in results) / len(results)
        else:
            pass_rate = 0.0

        # Return detailed results and overall pass rate
        return {
            "name": benchmark.name,
            "pass_rate": pass_rate,
            "results": results,
        }


def create_agent():
    """Create a new instance of TinyAgent"""
    return TinyAgent(
        llm=llm,
        memory=Memory(),
        tools=NativeTools(),
        planner=NativeReAct(),
    )

## 3 - Exact Match

In [4]:
import re

def exact_match_scorer(prediction: str, example: dict) -> bool:
    """Return True if the answer matches the prediction, False otherwise"""
    match = re.search(r"\b([A-J])\b", prediction.upper())
    return match.group(1) == example["expected"]

In [5]:
# Three examples from MMLU Pro
mmlu_pro = Benchmark(
    name="MMLU Pro",
    examples=[
        {
            "task": """Which of the following is the body cavity that contains the pituitary gland?
A) Ventral B) Dorsal C) Buccal D) Thoracic E) Pericardial F) Abdominal G) Spinal H) Pelvic I) Pleural J) Cranial
Answer with only the letter.""",
            "expected": "J",
        },
        {
            "task": """What is the approximate mean cranial capacity of Homo erectus?
A) 1200 cc B) under 650 cc C) 1700 cc D) 1350 cc E) just under 1000 cc F) 1500 cc G) under 500 cc H) about 800 cc I) just over 1100 cc J) about 900 cc
Answer with only the letter.""",
            "expected": "E",
        },
        {
            "task": """	
According to Moore’s “ideal utilitarianism,” the right action is the one that brings about the greatest amount of:
A) wealth. B) virtue. C) fairness. D) pleasure. E) peace. F) justice. G) happiness. H) power. I) good. J) knowledge.
Answer with only the letter.""",
            "expected": "I",
        },
    ],
    scorer=exact_match_scorer,
)

In [6]:
from rich import print

# Run evaluation
result = Evaluator(create_agent).run(mmlu_pro)
print(result)

{
    'name': 'MMLU Pro',
    'pass_rate': 0.3333333333333333,
    'results': [
        {'prediction': 'J', 'passed': True},
        {'prediction': 'I', 'passed': False},
        {'prediction': 'G', 'passed': False}
    ]
}

## 4 - Programmatic Check

In [7]:
def programmatic_scorer(prediction: str, example: dict) -> bool:
    """Check a prediction against its related check."""
    return example["check"](prediction)

In [8]:
# Three examples from IFeval
ifeval = Benchmark(
    name="IFeval",
    examples=[
        {
            "task": "Write me a funny song with less than 10 sentences for a proposal to build a new playground at my local elementary school.",
            "check": lambda text: sum(1 for c in text if c in ".!?") < 10,
        },
        {
            "task": "Write an ad copy for a new product, a digital photo frame that connects to your social media accounts and displays your photos. Respond with at most 150 words.",
            "check": lambda text: len(text.split()) <= 150,
        },
        {
            "task": "I am planning a trip to Japan, and I would like thee to write an itinerary for my journey in a Shakespearean style. You are not allowed to use any commas in your response.",
            "check": lambda text: "," not in text,
        },
    ],
    scorer=programmatic_scorer,
)

In [9]:
# Run evaluation
result = Evaluator(create_agent).run(ifeval)
print(result)

{
    'name': 'IFeval',
    'pass_rate': 1.0,
    'results': [
        {
            'prediction': '(To the tune of "Twinkle Twinkle Little Star")\n\nOur old swings are rusty and 
sad,\nWhere the perfect fun used to be had.\nThe monkey bars are mighty thin,\nAnd bored little feet just slip and 
spin.\nA brand new playground we implore,\nWith slides and swings and so much more!\nLet\'s build a place for 
joyful glee,\nFor every kid and you and me!',
            'passed': True
        },
        {
            'prediction': "**Title Idea: Your Memories, Beautifully Displayed. Always.**\n\nTired of physical photo
arrangements that quickly get outdated? Your most cherished moments shouldn't stay trapped on your phone or buried 
in social media feeds.\n\nIntroducing The Continuum Frame—the revolutionary digital photo frame that does the heavy
lifting for you. Simply connect your favorite social media accounts (Facebook, Instagram, etc.). Our intelligent 
technology automatically curates your photos, displaying the best moments from friends, family, and loved ones, all
in one elegant, rotating gallery.\n\nIt's like having a constant, personalized photo album updated in real-time, 
wherever life takes you. No printing required. No manual setup. Just pure, effortless beauty.\n\nStop scrolling 
through memories and start living them.\n\n**Get The Continuum Frame today and turn your digital life into lasting,
stunning decor.**\n*Available now at [YourWebsite.com].*",
            'passed': True
        },
        {
            'prediction': "Hark gentle traveler prepare thy spirit for Japan's embrace a journey most 
grand\n\n**DAY THE FIRST IN TOKYO'S LUMINOUS SHORE**\n\nThou shalt wake where neon doth fiercely gleam\nA labyrinth
of wonders a glorious scheme\nBehold Shibuya's crosswalk a river of souls\nWhere modern magic fiercely takes its 
tolls\nWander through Harajuku's painted delights\nSee fashion's grand pageant through day and the nights\nHark to 
a glimpse of temples old and serene\nSenso-ji's pagoda a timeless jade queen\n\n**DAY THE SECOND TOKYO'S SPIRITUAL 
HEART**\n\nThy morning quest shall be Imperial grounds so deep\nWhere ancient shadows guard secrets to 
keep\nThrough Meiji Jingu's gates a tranquil mist descends\nA forest's deep solace where earthly toil ends\nTaste 
artisan crafts a matcha so sweet\nWalk Ginza's pavements a shopper's grand feat\nBy dusk we travel by swift 
Shinkansen's might\nToward Kyoto's grace a softer pale light\n\n**DAY THE THIRD IN KYOTO'S QUIET DREAM**\n\nBehold 
Gion's cobbled ways a whispered delight\nWhere geiko wander bathed in lantern light\nStroll through Arashiyama 
bamboo groves so tall\nHear nature's soft music answering every call\nVisit Kiyomizu-dera a perch high above\nWhere
history sings its melodic love\nSeek Ryoan-ji's rocks a ponderous gaze\nLost in Zen silence through passing slow 
days\n\n**DAY THE FOUR A DAY OF GOLD AND OF PEARL**\n\nJourney hither to Nara a gentle return\nTo sacred wild deer 
whose sweet manners burn\nFeed creatures majestic by Todai-ji's keep\nWhere giant Buddha's slumber doth eternally 
sleep\nThen Kyoto calls for Fushimi's vermillion stream\nA thousand torii gates fulfilling a pilgrim's grand 
dream\nLet thy heart feel the echo of ages long past\nA memory of glory forever built to last\n\n**DAY THE FIFTH 
RETURN AND DEPARTURE'S SONG**\n\nOn morning bright we bid farewell with a heart full of cheer\nOur voyage concludes
but our memory remains dear\nReturn to a station thy parting we find\nLeaving Japan's splendor impressed on the 
mind\nGo hence gentle traveler with tales yet untold\nA grandeur of beauty more precious than gold\nFarewell until 
fate doth draw us together anew\nAdieu my dear pilgrim thy journey was true",
            'passed': True
        }
    ]
}

## 5 - LLM-as-a-judge

In [10]:
# Judge
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/", 
    api_key="MY_API_KEY"
)
judge = LLM(model="gemini-3.1-flash-lite", client=client, think=True)

In [11]:
def judge_scorer(prediction: str, example: dict) -> bool:
    """The LLM-as-a-judge scorer."""
    prompt = f"""
Score the response from 0.0 to 1.0.

Expected: {example["expected"]}
Response: {prediction}

Reply with only a single number.
"""
    response = judge.generate([{"role": "user", "content": prompt}])
    score = float(response.content.strip().split()[0])
    return score

In [12]:
# Three examples adapted from MMLU Pro
mmlu_pro = Benchmark(
    name="MMLU Pro",
    examples=[
        {
            "task": "Which body cavity contains the pituitary gland?",
            "expected": "the cranial cavity",
        },
        {
            "task": "What is the approximate mean cranial capacity of Homo erectus?",
            "expected": "just under 1000 cc",
        },
        {
            "task": "According to Moore's 'ideal utilitarianism,' the right action is the one that brings about the greatest amount of what?",
            "expected": "good",
        },
    ],
    scorer=judge_scorer,
)

In [13]:
# Run evaluation
result = Evaluator(create_agent).run(mmlu_pro)
print(result)

{
    'name': 'MMLU Pro',
    'pass_rate': 0.7999999999999999,
    'results': [
        {
            'prediction': 'The pituitary gland hangs within the **sella turcica** of the **sphenoid 
bone**.\n\nWhile the pituitary gland is housed within a specific bony indentation (the sella turcica), which is 
part of the skull, when discussing the major body cavities, it is situated near the boundaries of the **dura 
mater** that encloses the brain.\n\nTherefore, the most accurate general category is that it is within the 
**cranial cavity** (the skull cavity), but more specifically, it is physically housed in the sella turcica area.',
            'passed': 1.0
        },
        {
            'prediction': 'The approximate mean cranial capacity of *Homo erectus* is generally estimated to fall 
in the range of **600 to 900 cubic centimeters (cc)**.\n\nHowever, many paleoanthropologists and studies tend to 
cluster the average capacity closer to the **700 to 800 cc** range.\n\n### Key Considerations:\n\n1. **Variation is
High:** It is crucial to understand that this is a broad range. *Homo erectus* lived over a long period (from 
approximately 2 million years ago to 100,000 years ago) and across diverse geographical locations. Specimens vary 
greatly in size.\n2. **Comparison:**\n    * **Earlier Hominins (e.g., Australopithecus):** Much lower, typically 
300–550 cc.\n    * **Modern Humans (Homo sapiens):** Typically 1200–1500 cc.\n3. **Scientific Debate:** Due to 
differing techniques in calculating endocranial volume (the internal volume of the brain cavity) from fossil 
skulls, the exact mean estimate can shift depending on the specific metrics, which contributes to the wide accepted
range.',
            'passed': 0.4
        },
        {'prediction': '**Happiness** (or, more broadly, the greatest good/well-being).', 'passed': 1.0}
    ]
}

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

We added a single new module — `evaluator.py` — on top of the Chapter 6 agent. The agent itself did not change. That separation is the point: a clean boundary between the system under test and the system doing the testing lets you iterate on either side without breaking the other.

In [14]:
from illustrated_agents.chapters.ch7 import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py                                                                                                    │
│ ├── evaluator.py  ← New (Run a `TinyAgent` over a `Benchmark` suite and score outcomes.)                        │
│ ├── llm.py                                                                                                      │
│ ├── memory.py                                                                                                   │
│ ├── planning.py                                                                                                 │
│ ├── toolbox.py                                                                                                  │
│ ├── tools.py                                                                                                    │
│ └── trajectory.py                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# What's Next

Up next is Chapter 8 on Multi-Agent Collaboration.